# 🤖 Task 4: Context-Aware Chatbot Using LangChain + RAG

**Objective:** Build a conversational chatbot that remembers context and retrieves external information using RAG (Retrieval-Augmented Generation).

| Component | Technology |
|---|---|
| Knowledge Base | Built-in AI/ML corpus (10 topic documents) |
| Embeddings | `all-MiniLM-L6-v2` via HuggingFace (local, free) |
| Vector Store | FAISS (in-memory) |
| LLM | Mistral-7B-Instruct-v0.2 via HuggingFace InferenceClient (free) |
| Context Memory | Manual chat history injected into prompt |
| Deployment | Streamlit (`chatbot_app.py`) |

**Note:** This notebook demonstrates and tests each component of the RAG pipeline step by step.
The full deployable Streamlit app is in `chatbot_app.py`.

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers huggingface-hub streamlit python-dotenv tf-keras -q

## 📚 Step 2 — Define the Knowledge Base Corpus

In [ ]:
# Custom corpus — 10 AI/ML topic documents
# In a real project, replace these with your own PDFs, Wikipedia pages, or documents

CORPUS = [
    """
    Machine Learning Overview
    Machine learning (ML) is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed.
    The three main types: Supervised Learning (labeled data), Unsupervised Learning (unlabeled data),
    and Reinforcement Learning (reward-based agent training).
    """,
    """
    Deep Learning and Neural Networks
    Deep learning uses neural networks with many layers to learn data representations.
    Key architectures: CNNs (image recognition), RNNs/LSTMs (sequential data),
    Transformers (NLP, self-attention mechanism), GANs (generative models).
    """,
    """
    Natural Language Processing (NLP)
    NLP enables machines to understand human language.
    Key tasks: text classification, NER, machine translation, question answering,
    text summarization, sentiment analysis.
    Modern NLP uses pretrained models: BERT, GPT-4, LLaMA.
    """,
    """
    Large Language Models (LLMs) and BERT
    BERT: Bidirectional transformer encoder by Google (2018). 110M parameters.
    Pretrained on masked language modeling. Fine-tuned for classification, NER, QA.
    AG News: benchmark dataset with 120,000 samples across 4 categories (World/Sports/Business/Sci-Tech).
    GPT: Unidirectional transformer decoder by OpenAI. GPT-4 is multimodal.
    """,
    """
    Retrieval-Augmented Generation (RAG)
    RAG combines retrieval + generation to reduce hallucinations.
    Pipeline: 1) Index documents as vectors in FAISS.
    2) Retrieve top-k similar chunks for a query.
    3) Pass retrieved context + query to LLM for grounded answer generation.
    Benefits: reduces hallucinations, accesses up-to-date info, enables source attribution.
    """,
    """
    Scikit-learn and ML Pipelines
    Pipeline API chains preprocessing + model into one object (prevents data leakage).
    ColumnTransformer: applies different preprocessing to numeric vs categorical columns.
    GridSearchCV: exhaustive hyperparameter tuning with k-fold cross-validation.
    StandardScaler, OneHotEncoder (drop='first' avoids dummy variable trap).
    joblib: export trained pipelines for production reuse.
    """,
    """
    Model Evaluation Metrics
    Classification: Accuracy, Precision=TP/(TP+FP), Recall=TP/(TP+FN),
    F1=harmonic mean of precision+recall, ROC-AUC, Confusion Matrix.
    F1 and ROC-AUC are better than accuracy for imbalanced datasets.
    Regression: MAE, RMSE (penalizes large errors), R-squared.
    Cross-validation: k-Fold, Stratified k-Fold.
    """,
    """
    Transfer Learning and Fine-tuning
    Reuse a pretrained model as the starting point for a new task.
    Steps: pretrain on large data -> fine-tune on task-specific small data.
    Example: BERT fine-tuned on AG News (4-class classification).
    PEFT methods like LoRA (Low-Rank Adaptation) fine-tune only a small fraction
    of parameters, making it feasible on consumer hardware.
    """,
    """
    Customer Churn Prediction
    Telco Churn Dataset: 7,043 customers, 20 features, ~27% churn (imbalanced).
    Key predictors: Contract type, tenure, MonthlyCharges, InternetService.
    Use F1-score and ROC-AUC (not accuracy) due to imbalance.
    Approach: Logistic Regression + Random Forest with GridSearchCV.
    Both models in one GridSearchCV using list-style param_grid. Export with joblib.
    """,
    """
    Vector Databases and Embeddings
    Vector DBs store and search high-dimensional embeddings efficiently.
    FAISS: open-source, in-memory, extremely fast (Facebook AI).
    Pinecone: managed cloud vector DB. Chroma: lightweight, LangChain-friendly.
    Embedding models: all-MiniLM-L6-v2 (free, 80MB, local), all-mpnet-base-v2 (higher quality).
    Similarity: cosine similarity (angle between vectors), L2 distance (geometric).
    """
]

print(f'Corpus loaded: {len(CORPUS)} documents')
print(f'Sample document (first 200 chars):\n{CORPUS[0][:200]}')

## 🔪 Step 3 — Split Documents into Chunks

Large documents are split into smaller overlapping chunks so the retriever
can find the most relevant piece of text for each query.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in CORPUS]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

print(f'Documents  : {len(documents)}')
print(f'Chunks     : {len(chunks)}')
print(f'\nSample chunk:\n{chunks[0].page_content}')

## 🧠 Step 4 — Create Embeddings and Build FAISS Vector Store

`all-MiniLM-L6-v2` converts each text chunk into a 384-dimensional vector.
FAISS indexes these vectors for fast cosine similarity search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print('Loading embedding model (downloads ~80MB on first run)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)

print('Building FAISS vector store...')
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f'Vector store built with {len(chunks)} indexed chunks.')

# Test retrieval with a sample query
test_query = 'How does BERT work?'
retrieved = vectorstore.similarity_search(test_query, k=2)
print(f'\nTest query: "{test_query}"')
for i, doc in enumerate(retrieved, 1):
    print(f'\nRetrieved chunk {i}:\n{doc.page_content[:300]}')

## 💬 Step 5 — Set Up HuggingFace LLM (InferenceClient)

Enter your **free** HuggingFace token below.
Get one at: https://huggingface.co/settings/tokens

Make sure to enable **"Make calls to Inference Providers"** when creating the token.

In [ ]:
from huggingface_hub import InferenceClient

# Paste your HuggingFace token here
HF_TOKEN = 'hf_your_token_here'   # <-- replace with your token

RAG_PROMPT_TEMPLATE = """You are a helpful AI/ML assistant. Use the retrieved context below
to answer the question. If the context does not contain enough information, say so clearly.
Keep your answer concise and factual.

Conversation history:
{chat_history}

Retrieved context:
{context}

Question: {question}

Answer:"""


def call_llm(prompt: str) -> str:
    """Call Mistral-7B via HuggingFace InferenceClient chat_completion API."""
    client = InferenceClient(
        model='mistralai/Mistral-7B-Instruct-v0.2',
        token=HF_TOKEN,
    )
    response = client.chat_completion(
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=400,
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


print('LLM client configured (Mistral-7B-Instruct-v0.2 via HuggingFace free API)')

## 🔍 Step 6 — RAG Pipeline Function

Combines retrieval + context memory + LLM generation into one function.

In [ ]:
def answer_question(question: str, chat_history: list) -> tuple:
    """
    Full RAG pipeline:
    1. Retrieve top-3 relevant chunks from FAISS
    2. Inject last 6 turns of chat history (context memory)
    3. Build prompt and call Mistral-7B
    Returns (answer, source_chunks)
    """
    # Step 1: Retrieve
    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
    docs = retriever.invoke(question)
    context = '\n\n'.join([doc.page_content.strip() for doc in docs])

    # Step 2: Format chat history (context memory)
    history_text = ''
    for turn in chat_history[-6:]:   # keep last 6 turns
        history_text += f"Human: {turn['human']}\nAssistant: {turn['ai']}\n\n"

    # Step 3: Build prompt
    prompt = RAG_PROMPT_TEMPLATE.format(
        chat_history=history_text if history_text else 'None yet.',
        context=context,
        question=question,
    )

    # Step 4: Call LLM
    answer = call_llm(prompt)
    sources = [doc.page_content.strip() for doc in docs]
    return answer, sources

print('RAG pipeline function ready.')

## 🧪 Step 7 — Test the RAG Pipeline

Test single-turn and multi-turn (context memory) queries.

In [ ]:
# Single-turn test
chat_history = []

question1 = 'What is RAG and why is it useful?'
print(f'Q: {question1}')
answer1, sources1 = answer_question(question1, chat_history)
print(f'A: {answer1}')
print(f'\nRetrieved {len(sources1)} source chunks.')

# Save to chat history for multi-turn test
chat_history.append({'human': question1, 'ai': answer1})

In [ ]:
# Multi-turn test — follow-up question uses context memory
question2 = 'How does FAISS fit into this pipeline?'
print(f'Q: {question2}')
print('(Using context from previous question via chat history)')
answer2, sources2 = answer_question(question2, chat_history)
print(f'A: {answer2}')

chat_history.append({'human': question2, 'ai': answer2})
print(f'\nChat history now has {len(chat_history)} turns.')

In [ ]:
# Show retrieved source chunks for transparency
print('=== Retrieved Source Chunks for Last Query ===')
for i, src in enumerate(sources2, 1):
    print(f'\nChunk {i}:')
    print(src[:400])

## 🎯 Step 8 — Interactive Chat Loop (Terminal-Based)

A simple interactive loop for testing in the notebook.
Type `quit` or `exit` to stop. Type `clear` to reset history.

In [ ]:
# Run this cell for an interactive chat session in the notebook
# Press the Stop button or type 'quit' to end

chat_history = []
print('=== RAG Chatbot Interactive Mode ===')
print('Type your question. Type "quit" to exit, "clear" to reset history.')
print('=' * 45)

while True:
    try:
        user_input = input('\nYou: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\nSession ended.')
        break

    if not user_input:
        continue
    if user_input.lower() in ['quit', 'exit']:
        print('Goodbye!')
        break
    if user_input.lower() == 'clear':
        chat_history = []
        print('Chat history cleared.')
        continue

    try:
        answer, sources = answer_question(user_input, chat_history)
        print(f'\nBot: {answer}')
        print(f'[Retrieved {len(sources)} source chunk(s)]')
        chat_history.append({'human': user_input, 'ai': answer})
    except Exception as e:
        print(f'Error: {e}')

## 🚀 Step 9 — Deploy as Streamlit App

The full production chatbot with a proper web UI is in `chatbot_app.py`.
Run it with:

In [ ]:
# Uncomment and run this cell to launch the Streamlit app from the notebook
# Make sure chatbot_app.py is in the same directory

# import subprocess
# subprocess.Popen(['streamlit', 'run', 'chatbot_app.py'])
# print('Streamlit app launching at http://localhost:8501')

print('To launch the Streamlit app, run this in your terminal:')
print('  streamlit run chatbot_app.py')
print()
print('The app provides:')
print('  - Full chat UI with message history')
print('  - Sidebar token input and example questions')
print('  - Retrieved source chunk viewer per answer')
print('  - Clear history button')

## 📊 Step 10 — Summary

This notebook demonstrated a complete RAG pipeline:

| Step | What happened |
|---|---|
| Corpus | 10 AI/ML documents loaded as custom knowledge base |
| Chunking | `RecursiveCharacterTextSplitter` (500 chars, 50 overlap) |
| Embedding | `all-MiniLM-L6-v2` — 384-dim vectors, free, runs locally |
| Vector Store | FAISS — indexed all chunks for cosine similarity search |
| Retrieval | Top-3 most relevant chunks fetched per query |
| Context Memory | Last 6 conversation turns injected into every prompt |
| LLM | Mistral-7B-Instruct-v0.2 via HuggingFace InferenceClient (free) |
| Deployment | Streamlit chat UI (`chatbot_app.py`) |

**Skills demonstrated:** Conversational AI development · Document embedding · Vector search · RAG · LLM integration · Streamlit deployment